#  Business Intelligence Report: US Superstore Performance Analysis
**Role:** Senior Data Analyst  
**Objective:** Translate retail business objectives into actionable insights from the `superstore_dataset.csv` file, design professional diagnostic visualizations using Matplotlib/Seaborn, create interactive widgets with `ipywidgets` for dynamic analysis, and deliver a comprehensive executive summary with tactical strategic recommendations.

---

## 1. Data Definition and Preparation


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from ipywidgets import interact, Dropdown, IntSlider
import time
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# 1. Load the real dataset
try:
    df = pd.read_csv('superstore_dataset.csv', encoding='ISO-8859-1')
    print(" Success: 'superstore_dataset.csv' loaded perfectly.")
except FileNotFoundError:
    print(" Error: 'superstore_dataset.csv' not found. Please ensure the file is in the working directory.")

print(f"Initial Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")

### Data Cleaning and Feature Engineering Justifications:


In [ ]:
# Drop duplicates
df = df.drop_duplicates()

# Address missing values logically
if 'Postal Code' in df.columns:
    df['Postal Code'] = df['Postal Code'].fillna(0)
df = df.dropna(subset=['State'] if 'State' in df.columns else [])

# Standardize temporal dimensions
for date_col in ['Order Date', 'Ship Date']:
    if date_col in df.columns:
        df[date_col] = pd.to_datetime(df[date_col])

# Feature engineering
df['Profit Margin'] = (df['Profit'] / df['Sales']) * 100
df['Order Year'] = df['Order Date'].dt.year
df['Order Month'] = df['Order Date'].dt.month
df['Order Month-Year'] = df['Order Date'].dt.to_period('M')

print("Engineered variables preview:")
print(df[['Order Date', 'Sales', 'Profit', 'Profit Margin', 'Order Month-Year']].head())

## 2. Deep-Dive Exploratory Analysis (Matplotlib & Dynamic Widgets)


In [ ]:
# Aggregate time-series elements
monthly_sales = df.groupby(['Order Month-Year', 'Category'])['Sales'].sum().reset_index()
monthly_sales['Date'] = monthly_sales['Order Month-Year'].dt.to_timestamp()

def plot_monthly_sales(category='All'):
    plt.figure(figsize=(14, 5.5))
    if category == 'All':
        total_monthly = df.groupby('Order Month-Year')['Sales'].sum()
        plt.plot(total_monthly.index.to_timestamp(), total_monthly.values, 
                 marker='o', linewidth=2.5, color='darkblue', label='Total Net Corporate Revenue')
        plt.title('Monthly Sales Trend Performance - Combined Portfolio', fontsize=14, fontweight='bold')
    else:
        category_data = monthly_sales[monthly_sales['Category'] == category]
        plt.plot(category_data['Date'], category_data['Sales'], 
                 marker='o', linewidth=2.5, color='crimson', label=category)
        plt.title(f'Monthly Sales Trend Performance - Segment: {category}', fontsize=14, fontweight='bold')
        
    plt.xlabel('Timeline / Order Date', fontsize=11)
    plt.ylabel('Gross Revenue Stream ($)', fontsize=11)
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plt.show()

categories_list = ['All'] + list(df['Category'].unique())
interact(plot_monthly_sales, category=Dropdown(options=categories_list, value='All', description='Category:'));

### Regional Distribution Breakdown


In [ ]:
state_revenue = df.groupby('State')['Sales'].sum().sort_values(ascending=True)

def plot_top_states(top_n=10):
    plt.figure(figsize=(12, max(4, top_n * 0.4)))
    selected_states = state_revenue.tail(top_n)
    
    bars = plt.barh(range(len(selected_states)), selected_states.values, color='teal', edgecolor='black', alpha=0.75)
    plt.yticks(range(len(selected_states)), selected_states.index)
    plt.xlabel('Cumulative Sales Volume ($)', fontsize=11)
    plt.ylabel('US State Territory', fontsize=11)
    plt.title(f'Top {top_n} Performing State Markets by Revenue', fontsize=14, fontweight='bold')
    
    for bar in bars:
        width = bar.get_width()
        plt.text(width + (state_revenue.max() * 0.01), bar.get_y() + bar.get_height()/2, 
                 f'${width:,.0f}', va='center', ha='left', fontsize=9, fontweight='bold', color='black')
        
    plt.grid(axis='x', linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.show()

interact(plot_top_states, top_n=IntSlider(min=5, max=25, value=10, description='Top N States:'));

## 3. Communicating Strategic Insights (Seaborn Diagnostic Analytics)


In [ ]:
top_products_profit = df.groupby('Product Name')['Profit'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 6))
ax = sns.barplot(x=top_products_profit.values, y=top_products_profit.index, palette='magma', orient='h')
plt.title('Top 10 Most Profitable Individual Inventory Products', fontsize=15, fontweight='bold', pad=15)
plt.xlabel('Total Generated Net Profit ($)', fontsize=11, fontweight='bold')
plt.ylabel('Product Technical Name', fontsize=11, fontweight='bold')

for idx, value in enumerate(top_products_profit.values):
    ax.text(value + (top_products_profit.max() * 0.01), idx, f'${value:,.0f}', va='center', fontweight='bold', fontsize=9)

plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

### Discount Vulnerability Matrix

In [ ]:
plt.figure(figsize=(13, 7.5))
sns.scatterplot(data=df, x='Discount', y='Profit', hue='Category', alpha=0.6, s=70, palette='Set1')
sns.regplot(data=df, x='Discount', y='Profit', scatter=False, color='darkred', line_kws={'linewidth': 2.5, 'linestyle': '--'})
plt.axhline(y=0, color='black', linestyle='-', alpha=0.5, linewidth=1.5)

plt.text(df['Discount'].max() * 0.45, df['Profit'].max() * 0.02, 'Break-Even Threshold Line', color='black', fontsize=11, fontweight='bold')
plt.title('Discount Optimization Matrix: Operational Margin Erosion Diagnostic', fontsize=15, fontweight='bold', pad=20)
plt.xlabel('Applied Discount Rate (%)', fontsize=11, fontweight='bold')
plt.ylabel('Net Transactional Profit / Loss ($)', fontsize=11, fontweight='bold')
plt.legend(title='Product Portfolio Segment', loc='upper right')
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

## 4. Methodology and Library Benchmarking Analysis

In [ ]:
print("=== LIBRARY PROCESSING VELOCITY BENCHMARK ===")

start_mt = time.time()
plt.figure(figsize=(4, 2))
plt.plot(df.groupby('Order Year')['Sales'].sum())
plt.close()
matplotlib_duration = time.time() - start_mt

start_sb = time.time()
plt.figure(figsize=(4, 2))
sns.lineplot(data=df.groupby('Order Year')['Sales'].sum().reset_index(), x='Order Year', y='Sales')
plt.close()
seaborn_duration = time.time() - start_sb

print(f"• Matplotlib primitive execution runtime: {matplotlib_duration:.4f} seconds")
print(f"• Seaborn structured wrapper runtime:     {seaborn_duration:.4f} seconds\n")

print("Core Architectural Recommendations:")
print("1. For iterative experimentation and fast widget interactions, Matplotlib is ideal because it operates with minimal processing overhead.")
print("2. For formal stakeholder reports, Seaborn is preferred because its built-in statistical charts and polished design elements help clearly communicate complex data insights.")

## 5. Executive Summary and Strategic Recommendations


In [ ]:
gross_revenue = df['Sales'].sum()
net_profit = df['Profit'].sum()
global_margin = (net_profit / gross_revenue) * 100
loss_making_discounts = (df[df['Discount'] > 0.2]['Profit'] < 0).mean() * 100


print("             EXECUTIVE C-SUITE BRIEFING - REAL TIME METRICS              ")

print(f" GLOBAL CORPORATE STATUS:")
print(f"  • Cumulative Enterprise Revenue: ${gross_revenue:,.2f}")
print(f"  • Consolidated Net Financial Profit: ${net_profit:,.2f}")
print(f"  • Structural Profit Margin: {global_margin:.2f}%")

print(f"\n RISK PROFILE ANALYSIS:")
print(f"  • Critical Promo Vulnerability: {loss_making_discounts:.1f}% of transactions with over 20% discount result in net financial losses.")

print("\n ACTIONABLE STRATEGIC RECOVERY RECOMMENDATIONS:")
print("  1. Implement a system-wide rule limiting standard customer discounts to a maximum of 20% on low-margin products (like Furniture).")
print("  2. Establish an automated multi-stage manager sign-off process for any exceptions that go over the 20% discount threshold.")


### Advanced Portfolio Unified Dashboard


In [ ]:
def render_unified_dashboard():
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    
    # Panel 1: Operational timeline
    monthly_total = df.groupby('Order Month-Year')['Sales'].sum()
    ax1.plot(monthly_total.index.to_timestamp(), monthly_total.values, marker='o', color='darkcyan', linewidth=2)
    ax1.set_title('Historical Monthly Revenue Volatility', fontsize=12, fontweight='bold')
    ax1.tick_params(axis='x', rotation=25)
    
    # Panel 2: Segment Breakdown
    cat_sales = df.groupby('Category')['Sales'].sum()
    ax2.bar(cat_sales.index, cat_sales.values, color=['#34495e', '#3498db', '#e74c3c'], alpha=0.85, edgecolor='black')
    ax2.set_title('Revenue Allocation by Product Category', fontsize=12, fontweight='bold')
    
    # Panel 3: Key States
    top_states_dash = state_revenue.tail(10)
    ax3.barh(range(len(top_states_dash)), top_states_dash.values, color='indigo', alpha=0.7)
    ax3.set_yticks(range(len(top_states_dash)))
    ax3.set_yticklabels(top_states_dash.index)
    ax3.set_title('Top 10 High-Performing Regional Markets', fontsize=12, fontweight='bold')
    
    # Panel 4: Policy Impact Mapping
    for cat_name in df['Category'].unique():
        sub_data = df[df['Category'] == cat_name]
        ax4.scatter(sub_data['Discount'], sub_data['Profit'], label=cat_name, alpha=0.4, s=30)
    ax4.axhline(y=0, color='red', linestyle='--', alpha=0.6)
    ax4.set_title('Discount Threshold vs Net Profitability Matrix', fontsize=12, fontweight='bold')
    ax4.set_xlabel('Discount Rate')
    ax4.set_ylabel('Net Profit')
    ax4.legend()
    
    plt.suptitle('Unified Business Intelligence Enterprise Dashboard - US Superstore Performance', fontsize=15, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.show()

render_unified_dashboard()